# TASHA activity-generation distributions (frequency, start time, duration, joint episodes, location bands)

Regenerate the empirical TASHA distributions from the estimation half of the RTS and compare them with the files used by the 2024 runs.

Not a Biogeme model: empirical distributions from the estimation half of the survey. Sections: 1. raw data,
2. building the distributions, 3. comparison with the files used by the model runs, 4. the distributions as charts.

In [ ]:
RUN_BUILD = False   # True: rebuild the distributions into output/ (about 10 s)
import os, sys, json, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
FOLDER = os.path.abspath("")
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
REF = os.path.join(FOLDER, "data", "reference_outputs")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Raw data

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

**2017/18 Regional Travel Survey (Baltimore and Washington regions), household travel survey, weekday travel day** — Baltimore Metropolitan Council (BMC) and the National Capital Region Transportation Planning Board (TPB) at the Metropolitan Washington Council of Governments (MWCOG)
- Where: <https://www.baltometro.org/ and https://www.mwcog.org/ (verify: no download page is documented in the project; the data are released on request)>
- How: Request the 2017/18 Regional Travel Survey public-use files from BMC (Baltimore region) or MWCOG/TPB (Washington region). The project used one flat file in which every trip row carries its household and person fields (household, person, trip and vehicle tables joined): RTS_raw_flat_data.csv, with the codebook RTS_codebook_data_dictionary.xlsx (sheets Household, Person, Trip, Vehicle). The two cleaned files rts_trips_clean.csv (163,290 trips, deduplicated) and rts_tours.csv (57,627 home-based tours) were made by 01_clean_rts.py of the tour-based MITO project; that script is not on this machine, so these two files are shipped and cannot be rebuilt here. Survey weights: wthhfin, wtperfin, wttrdfin. If you receive separate household / person / trip / vehicle tables instead, join them on household_id, person_id (persno) and tripid to the flat layout of the codebook.
- Put in: `00_shared_data/raw/rts_2017_18 (the project's copies are provided in 00_shared_data/rts_2017_18/raw as .csv.gz)`
- Files: `RTS_raw_flat_data.csv`, `RTS_codebook_data_dictionary.xlsx`, `rts_trips_clean.csv`, `rts_tours.csv`

**MSTM zone system: SMZ/RMZ polygons, zone table, ABIT zone attributes** — MDOT (MSTM), via ABIT/input/mito_los/Zonal and 2_silo/input/zoneSystem.csv; ABIT/input/maryland/zone_attributes.csv (ABIT Maryland set-up)
- Where: <https://www.roads.maryland.gov/ (verify: provided with the MSTM)>
- How: Zone polygons SMZ_RMZ_02152011inMSTM_EPSG26918.shp (+ .dbf .shx .prj; field id / SMZRMZ, 1,588 zones), zoneSystem.csv (ZoneId, COUNTYFIPS, ...), zone_attributes.csv (id, centroid x/y EPSG:26985, countyfips, region types, bbsr, area_km2; built for ABIT from mstm_26985.shp and zoneSystem.csv, see 3_abit/coef_respec/PROVENANCE.md, script not in the project).
- Put in: `00_shared_data/raw/mstm_zone_system`
- Files: `SMZ_RMZ_02152011inMSTM_EPSG26918.shp/.dbf/.shx/.prj`, `zoneSystem.csv`, `zone_attributes.csv`

**Census 2010 Gazetteer file of census tracts (internal points)** — U.S. Census Bureau
- Where: <https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html (verify: the project copy I695_pricing/data/census/Gaz_tracts_national.txt has no download log)>
- How: 2010 Gazetteer, Census Tracts, national file Gaz_tracts_national.txt (GEOID, INTPTLAT, INTPTLONG). Used by survey_population.tract_zone: zone of each tract's internal point.
- Put in: `00_shared_data/raw/census_gazetteer_2010`
- Files: `Gaz_tracts_national.txt`

## 2. Data preparation

Shared and upstream stages (outputs provided in `../00_shared_data` and `input/data/raw`; run them only for a full rebuild from the raw data; `cd` paths from this model folder):

- **Step 1.** shared preparation S00: Download / request the raw data (DATA_SOURCES.json) into 00_shared_data/raw/<folder> (one README per folder says what goes there)  
  `cd ../00_shared_prep/code` → `sh fetch_gtfs.sh && sh fetch_gtfs_regional.sh   # the GTFS feeds; the other sources are manual downloads or requests`
- **Step 2.** shared preparation S01: Survey cleaning: rts_trips_clean.csv and rts_tours.csv from the RTS flat file (01_clean_rts.py of the tour-based MITO project)  
  `cd ../00_shared_prep/code` → `none: the cleaning script is not on this machine; use the provided files`
- **Step 3.** shared preparation S02: Estimation / hold-out split of the survey households (seed 2026, stratified by county x household size)  
  `cd ../00_shared_prep/code` → `PYTHONDONTWRITEBYTECODE=1 ~/.venvs/biogeme/bin/python make_rts_split.py`
- **Step 4.** shared preparation S03: Tract internal point -> model zone (tract_zone.csv) and persons per zone in 2016 (zone_population_2016.csv)  
  `cd ../00_shared_prep/code` → `PYTHONDONTWRITEBYTECODE=1 python3 build_survey_population.py --rebuild-zone-files   # also runs S04; needs geopandas (project Python)`
- **Step 5.** shared preparation S04: Survey population of both halves in the ABIT input format (households, persons, dwellings, person map, workplaces)  
  `cd ../00_shared_prep/code` → `PYTHONDONTWRITEBYTECODE=1 ~/.venvs/biogeme/bin/python build_survey_population.py   # uses the provided tract_zone.csv / zone_population_2016.csv; add --rebuild-zone-files for S03`
- **Step 6.** shared preparation S05: Typed survey episodes of both halves (TASHA activity types, person segments, zones of the trip ends)  
  `cd ../00_shared_prep/code` → `PYTHONDONTWRITEBYTECODE=1 ~/.venvs/biogeme/bin/python build_episodes.py`
- **Step 7.** build the distributions into output/  
  `cd code` → `PYTHONDONTWRITEBYTECODE=1 TASHA_JOINT_PB=1 TASHA_LIC_TYPES=IO_ACC,IS,IO_REC,IO_OTH python3 tasha_distributions.py`
- **Step 8.** compare with data/reference_outputs  
  `cd code` → `python3 compare_with_reference.py`

Package steps, run below when `RUN_DATA_PREP = True`: they rebuild `input/data/estimation` and `input/data/holdout`; `check_against_provided.py` compares the rebuilt tables with the provided ones.

**Step 7.** build the distributions into output/

In [ ]:
if RUN_BUILD:
    nt.run('PYTHONDONTWRITEBYTECODE=1 TASHA_JOINT_PB=1 TASHA_LIC_TYPES=IO_ACC,IS,IO_REC,IO_OTH python3 tasha_distributions.py', cwd=os.path.join(FOLDER, "code"))
else:
    print("skipped (RUN_BUILD = False)")

**Step 8.** compare with data/reference_outputs

In [ ]:
if RUN_BUILD:
    nt.run('python3 compare_with_reference.py', cwd=os.path.join(FOLDER, "code"))
else:
    print("skipped (RUN_BUILD = False)")

## 3. Files

`data/reference_outputs` holds the files used by the model runs; `output/` your rebuild.

In [ ]:
display(nt.files_table(REF))
src = os.path.join(FOLDER, "output") if glob.glob(os.path.join(FOLDER, "output", "tasha_*.csv")) else REF
print("charts below from:", os.path.relpath(src, FOLDER))
def probs(t):
    return np.vstack(t["probs"].map(lambda s: np.array([float(x) for x in str(s).split(";")])).to_numpy())

## 4. The distributions

### Frequency P(n | type, segment, household, prior): mean episodes per person-day

In [ ]:
fq = pd.read_csv(os.path.join(src, "tasha_frequency.csv"))
fq = fq[fq["level"] == fq["level"].iloc[0]] if "level" in fq else fq
m = fq.groupby(["type", "seg"]).apply(lambda g: np.average(g["mean"], weights=g["n"])).unstack()
fig, ax = plt.subplots(figsize=(1.1 * m.shape[1] + 3, 0.45 * m.shape[0] + 1.5))
im = ax.imshow(m.values, cmap="Blues"); plt.colorbar(im, ax=ax, label="mean episodes")
ax.set_xticks(range(m.shape[1])); ax.set_xticklabels(m.columns, rotation=45, ha="right"); ax.set_yticks(range(m.shape[0])); ax.set_yticklabels(m.index)
for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        if np.isfinite(m.values[i, j]): ax.text(j, i, f"{m.values[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if m.values[i, j] > np.nanmax(m.values) * 0.6 else "black")
ax.set_title("Mean episodes per person-day (weighted by cell size)"); plt.tight_layout(); plt.show()

### Start time P(t | type, …): 15-minute bins from midnight, averaged over the cells of each type

In [ ]:
stt = pd.read_csv(os.path.join(src, "tasha_start.csv"))
P = probs(stt); hours = np.arange(P.shape[1]) * 0.25
types = sorted(stt["type"].unique())
fig, axes = plt.subplots(int(np.ceil(len(types) / 4)), 4, figsize=(15, 2.3 * np.ceil(len(types) / 4)), sharex=True)
for ax, t in zip(axes.flat, types):
    i = (stt["type"] == t).to_numpy(); w = stt.loc[i, "n"].to_numpy()
    ax.fill_between(hours, (P[i] * w[:, None]).sum(0) / w.sum(), step="mid", color="#2b6cb0", alpha=0.8); ax.set_title(t, fontsize=9)
for ax in axes.flat[len(types):]: ax.axis("off")
fig.supxlabel("start (hour of day)"); plt.tight_layout(); plt.show()

### Duration P(δ | type, …): 5-minute bins to 2 h, then 15 minutes

In [ ]:
du = pd.read_csv(os.path.join(src, "tasha_duration.csv"))
edges = np.array(list(range(0, 121, 5)) + list(range(135, 1441, 15)), float); mids = (edges[:-1] + edges[1:]) / 2 / 60
P = probs(du)[:, :len(mids)]
types = sorted(du["type"].unique())
fig, axes = plt.subplots(int(np.ceil(len(types) / 4)), 4, figsize=(15, 2.3 * np.ceil(len(types) / 4)), sharex=True)
for ax, t in zip(axes.flat, types):
    i = (du["type"] == t).to_numpy(); w = du.loc[i, "n"].to_numpy()
    dens = (P[i] * w[:, None]).sum(0) / w.sum() / np.diff(edges) * 60
    ax.plot(mids, dens, color="#2b6cb0"); ax.set_xlim(0, 12); ax.set_title(t, fontsize=9)
for ax in axes.flat[len(types):]: ax.axis("off")
fig.supxlabel("duration (hours)"); fig.supylabel("density per hour"); plt.tight_layout(); plt.show()

### Joint episodes and location bands

In [ ]:
for f in ["tasha_joint_propensity.csv", "tasha_joint_propensity_mand.csv", "tasha_joint_participants.csv", "tasha_location_bands.csv", "tasha_params.csv"]:
    p = os.path.join(src, f)
    if os.path.exists(p):
        print(f); display(pd.read_csv(p))